**Модель:** `ai-forever/rugpt3small_based_on_gpt2`  
**Датасет:** `ScoutieAutoML/medical_news_dataset`

Решение:

1. Загружаем русскоязычные медицинские новости.
2. Убираем рекламу, ссылки и слишком короткие тексты.
3. Дообучаем на задаче продолжения медицинской новости.
4. Сравниваем несколько режимов `model.generate`.
5. Подбираем параметры генерации.
6. Добавляем `StoppingCriteria`, чтобы новость не обрывалась внутри предложения.
7. Генерируем 10 учебных синтетических примеров.

Все итоговые примеры явно помечаются как синтетические. В промптах используются вымышленные учреждения и темы медицинского образования.

Сравниваются:

- **Greedy Search** — выбирается наиболее вероятный следующий токен;
- **Beam Search** — одновременно рассматривается несколько вариантов продолжения;
- **Temperature** — регулирует случайность генерации;
- **Top-k** — оставляет только `k` наиболее вероятных токенов;
- **Top-p** — оставляет минимальный набор токенов с суммарной вероятностью `p`;
- **repetition_penalty** — уменьшает количество повторов;
- **no_repeat_ngram_size** — запрещает повторение одинаковых n-грамм.

Для итоговой генерации используются:

```text
temperature = 0.80
top_k = 50
top_p = 0.92
repetition_penalty = 1.12
no_repeat_ngram_size = 3
```

Дополнительно используется собственный `SentenceEndStoppingCriteria`. После минимальной длины он разрешает остановку только на `.`, `!` или `?`. Если одного блока токенов не хватило, генерация продолжается следующим вызовом `model.generate`.

In [1]:
%pip install -q -U transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.9 MB/s eta 0:00:00


In [2]:
# Импорты

import math
import random
import re

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    StoppingCriteria,
    StoppingCriteriaList,
    Trainer,
    TrainingArguments,
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [14]:
#===
# Загрузка и фильтрация датасета

DATASET_NAME = "ScoutieAutoML/medical_news_dataset"

# Загружаем датасет заново,
# чтобы dataset точно не был пустым
dataset = load_dataset(
    DATASET_NAME,
    split="train",
)

print(
    "Всего строк:",
    len(dataset)
)

print(
    "language:",
    dataset.unique("language")
)

print(
    "spam:",
    dataset.unique("spam")
)


#===
# Фильтрация

def keep_row(row):
    text = row.get("text")

    if text is None:
        return False

    text = clean_text(
        text
    )

    language = str(
        row.get(
            "language",
            ""
        )
    ).strip().lower()

    spam = str(
        row.get(
            "spam",
            ""
        )
    ).strip().upper()

    return (
        language == "rus"
        and spam == "NOT SPAM"
        and 180 <= len(text) <= 3500
    )


dataset = dataset.filter(
    keep_row
)


print(
    "После фильтрации:",
    len(dataset)
)


if len(dataset) == 0:
    raise RuntimeError(
        "После фильтрации датасет пустой"
    )


print()
print(
    "Пример текста:"
)

print(
    dataset[0]["text"][:500]
)

Всего строк: 22306
language: ['rus', 'other', 'eng']
spam: ['NOT SPAM', 'SPAM']


Filter:   0%|          | 0/22306 [00:00<?, ? examples/s]

После фильтрации: 13214

Пример текста:
Пациентка с очень  высоким риском  сердечно-сосудистых осложнений (инфаркт, инсульт и прочих неприятностей). Для таких пациентов норма ЛПНП ниже 1,4 ммоль/л, общий холестерин менее 4.00 ммоль/л.
Сделала перерыв в статинах, аллопуриноле (препарат для снижения мочевой кислоты). Вижу и кушала вкусно, сладко. На эхо-кг ухудшение по клапанам.
Вопрос риторический: зачем так делать?
Совет дня: следите пожалуйста как ваши старшие родственники принимают препараты
♥️
Это может сильно продлить их жизнь и е


In [15]:
# Очистка текста

def clean_text(text):
    text = str(text)

    text = re.sub(
        r"https?://\S+",
        " ",
        text,
    )

    text = re.sub(
        r"t\.me/\S+",
        " ",
        text,
    )

    text = re.sub(
        r"#\S+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()

In [16]:
# Ограничение выборки

MAX_SAMPLES = 6000

dataset = dataset.shuffle(
    seed=SEED
)

if len(dataset) > MAX_SAMPLES:
    dataset = dataset.select(
        range(MAX_SAMPLES)
    )

print(
    "Используем текстов:",
    len(dataset),
)

Используем текстов: 6000


In [17]:
# RuGPT

MODEL_NAME = (
    "ai-forever/"
    "rugpt3small_based_on_gpt2"
)

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_NAME
    )
)

model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_NAME
    )
)

model.config.pad_token_id = (
    tokenizer.pad_token_id
)

print(
    "Модель:",
    MODEL_NAME,
)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: ai-forever/rugpt3small_based_on_gpt2


In [18]:
# Текст для обучения

def prepare_text(row):
    text = clean_text(
        row["text"]
    )

    return {
        "train_text": (
            "Медицинская новость: "
            + text
            + tokenizer.eos_token
        )
    }


dataset = dataset.map(
    prepare_text
)


print(
    dataset[0][
        "train_text"
    ][:500]
)

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Медицинская новость: Клещи, они еще активны? Лето прошло, жара отступила и многие успокоились, считая, что сезон кровососущих паразитов, т.е. клещей позади. Но, расслабляться рано! В Европейской части России, два пика активности клещей – апрель-май и август-сентябрь, причем осенью активность клещей напрямую зависит от температуры воздуха и сохраняется до значений +6С. «Осенью у клещей открывается второе дыхание и это объяснимо, комфортнее всего они ощущают себя при влажности воздуха более 80%. О


In [19]:
# Train / Validation

split = dataset.train_test_split(
    test_size=0.10,
    seed=SEED,
)

train_dataset = split["train"]
val_dataset = split["test"]

print(
    "TRAIN:",
    len(train_dataset),
)

print(
    "VAL:",
    len(val_dataset),
)

TRAIN: 5400
VAL: 600


In [20]:
# Токенизация

MAX_LENGTH = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["train_text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_tokenized = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=(
        train_dataset.column_names
    ),
)

val_tokenized = val_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=(
        val_dataset.column_names
    ),
)

Map:   0%|          | 0/5400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

In [21]:
# Labels = input_ids

data_collator = (
    DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )
)

In [22]:
# Параметры обучения

OUTPUT_DIR = (
    "/content/"
    "rugpt_medical_news"
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=1,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

In [23]:
# Экономия памяти

model.gradient_checkpointing_enable()
model.config.use_cache = False

In [24]:
# Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ],
)

In [25]:
# Обучение

trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.258655,3.083350
2,3.085925,3.053801


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=676, training_loss=3.212253107827091, metrics={'train_runtime': 584.9511, 'train_samples_per_second': 18.463, 'train_steps_per_second': 1.156, 'total_flos': 1238249670912000.0, 'train_loss': 3.212253107827091, 'epoch': 2.0})

In [26]:
# Validation

metrics = trainer.evaluate()

eval_loss = metrics[
    "eval_loss"
]

perplexity = math.exp(
    eval_loss
)

print(
    f"eval_loss: {eval_loss:.4f}"
)

print(
    f"perplexity: {perplexity:.2f}"
)

Training Loss,Validation Loss,Epoch
3.085925,3.053801,2


eval_loss: 3.0538
perplexity: 21.20


In [27]:
# Режим генерации

model.config.use_cache = True
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50264, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50264, bias=False)
)

In [28]:
# Параметры generate

GENERATION_CONFIGS = {
    "Greedy": {
        "do_sample": False,
        "num_beams": 1,
    },

    "Beam Search": {
        "do_sample": False,
        "num_beams": 4,
        "early_stopping": True,
        "repetition_penalty": 1.10,
        "no_repeat_ngram_size": 3,
    },

    "Temperature": {
        "do_sample": True,
        "temperature": 1.15,
        "top_k": 0,
        "top_p": 1.0,
        "repetition_penalty": 1.05,
    },

    "Top-k + Top-p": {
        "do_sample": True,
        "temperature": 0.80,
        "top_k": 50,
        "top_p": 0.92,
        "repetition_penalty": 1.12,
        "no_repeat_ngram_size": 3,
    },
}

In [33]:
#===
# Остановка по . ! ?

class SentenceEndStoppingCriteria(
    StoppingCriteria
):
    def __init__(
        self,
        tokenizer,
        prompt_length,
        min_new_tokens=50,
    ):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length
        self.min_new_tokens = min_new_tokens

    def __call__(
        self,
        input_ids,
        scores,
        **kwargs,
    ):
        stops = []

        for row in input_ids:
            generated_ids = row[
                self.prompt_length:
            ]

            # Не останавливаем слишком рано
            if len(generated_ids) < self.min_new_tokens:
                stops.append(False)
                continue

            # Токены -> текст
            text = (
                self.tokenizer
                .decode(
                    generated_ids,
                    skip_special_tokens=True,
                )
                .rstrip()
            )

            # Конец полного предложения
            stops.append(
                bool(text)
                and text[-1] in ".!?"
            )

        # ВАЖНО:
        # возвращаем одномерный тензор (batch,)
        return torch.tensor(
            stops,
            dtype=torch.bool,
            device=input_ids.device,
        )

In [34]:
# Оставляем последнее полное предложение

def trim_to_sentence(text):
    text = text.strip()

    last_position = max(
        text.rfind("."),
        text.rfind("!"),
        text.rfind("?"),
    )

    if last_position >= 0:
        return text[
            :last_position + 1
        ].strip()

    return (
        text.rstrip(
            " ,;:-"
        )
        + "."
    )

In [35]:
# Генерация без обрыва

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model.to(DEVICE)

def generate_complete_news(
    prompt,
    generation_config,
    min_new_tokens=60,
    first_chunk=140,
    continue_chunk=80,
    max_rounds=3,
):
    prefix = (
        "Медицинская новость: "
        + prompt.strip()
    )

    input_ids = (
        tokenizer(
            prefix,
            return_tensors="pt",
        )["input_ids"]
        .to(DEVICE)
    )

    original_prompt_length = (
        input_ids.shape[1]
    )

    current_ids = input_ids

    for round_index in range(
        max_rounds
    ):
        current_prompt_length = (
            current_ids.shape[1]
        )

        minimum = (
            min_new_tokens
            if round_index == 0
            else 12
        )

        stopping = (
            StoppingCriteriaList(
                [
                    SentenceEndStoppingCriteria(
                        tokenizer=tokenizer,
                        prompt_length=(
                            current_prompt_length
                        ),
                        min_new_tokens=minimum,
                    )
                ]
            )
        )

        chunk_size = (
            first_chunk
            if round_index == 0
            else continue_chunk
        )

        kwargs = dict(
            generation_config
        )

        bad_words_ids = None

        if tokenizer.eos_token_id is not None:
            bad_words_ids = [
                [
                    tokenizer.eos_token_id
                ]
            ]

        with torch.no_grad():
            current_ids = model.generate(
                current_ids,
                max_new_tokens=chunk_size,
                min_new_tokens=minimum,
                pad_token_id=(
                    tokenizer.pad_token_id
                ),
                bad_words_ids=bad_words_ids,
                stopping_criteria=stopping,
                **kwargs,
            )

        generated_text = (
            tokenizer
            .decode(
                current_ids[
                    0,
                    original_prompt_length:
                ],
                skip_special_tokens=True,
            )
            .strip()
        )

        if (
            generated_text
            and generated_text[-1]
            in ".!?"
        ):
            break

    generated_text = (
        trim_to_sentence(
            generated_text
        )
    )

    full_text = (
        prefix
        + generated_text
    )

    if not full_text.endswith(
        (
            ".",
            "!",
            "?",
        )
    ):
        full_text += "."

    return (
        "[УЧЕБНАЯ СИНТЕТИЧЕСКАЯ НОВОСТЬ]\n"
        + full_text
    )

In [36]:
# Сравнение generate

compare_prompt = (
    "В вымышленном медицинском колледже "
    "«Академия здоровья» открыли новый "
    "симуляционный класс для студентов. "
)

comparison = []

for name, config in (
    GENERATION_CONFIGS.items()
):
    text = generate_complete_news(
        compare_prompt,
        generation_config=config,
        min_new_tokens=45,
        first_chunk=120,
    )

    comparison.append(
        {
            "режим": name,
            "текст": text,
            "корректное окончание": (
                text.rstrip()[-1]
                in ".!?"
            ),
        }
    )

comparison_df = pd.DataFrame(
    comparison
)

print(
    comparison_df.to_string(
        index=False
    )
)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


        режим                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         текст  корректное окончание
       Greedy                                                                                                                                                                                                                                                               [УЧЕБНАЯ СИНТЕТИЧЕСКАЯ НОВОСТЬ]\nМедицинская новость: В вымышленном медицинском колледже «Академия здоровья» открыли новый симуляционный к

In [37]:
# Лучшие параметры

FINAL_GENERATION_CONFIG = (
    GENERATION_CONFIGS[
        "Top-k + Top-p"
    ]
)

print(
    FINAL_GENERATION_CONFIG
)

{'do_sample': True, 'temperature': 0.8, 'top_k': 50, 'top_p': 0.92, 'repetition_penalty': 1.12, 'no_repeat_ngram_size': 3}


In [38]:
# 10 безопасных промптов

PROMPTS = [
    (
        "В вымышленном медицинском колледже "
        "«Академия здоровья» преподаватели "
        "представили новый симуляционный класс. "
    ),

    (
        "Студенты вымышленного медицинского "
        "института «МедПрофи» начали изучать "
        "анатомию с помощью виртуальной лаборатории. "
    ),

    (
        "В учебном центре «МедНавык» прошла "
        "неделя практических занятий для будущих "
        "медицинских сестер. "
    ),

    (
        "В вымышленной академии «Гиппократ Next» "
        "создали цифровой тренажер для отработки "
        "коммуникации с пациентами. "
    ),

    (
        "Студенческий научный кружок вымышленного "
        "колледжа представил проект интерактивного "
        "атласа человеческого организма. "
    ),

    (
        "В симуляционном центре вымышленного "
        "университета прошли соревнования студентов "
        "по оказанию первой помощи на манекенах. "
    ),

    (
        "Преподаватели вымышленного медицинского "
        "колледжа разработали новый курс по истории "
        "медицины для первокурсников. "
    ),

    (
        "В учебной клинике вымышленной академии "
        "открылась лаборатория медицинской "
        "коммуникации и этики. "
    ),

    (
        "Будущие фельдшеры вымышленного колледжа "
        "приняли участие в образовательной игре "
        "по работе бригады скорой помощи. "
    ),

    (
        "В вымышленном центре медицинского "
        "образования запустили виртуальный музей "
        "истории сестринского дела. "
    ),
]

In [39]:
# Генерация 10 новостей

generated_news = []

for index, prompt in enumerate(
    PROMPTS,
    start=1,
):
    text = generate_complete_news(
        prompt=prompt,
        generation_config=(
            FINAL_GENERATION_CONFIG
        ),
        min_new_tokens=55,
        first_chunk=150,
        continue_chunk=80,
        max_rounds=3,
    )

    generated_news.append(
        {
            "№": index,
            "prompt": prompt,
            "generated": text,
            "ends_correctly": (
                text.rstrip()[-1]
                in ".!?"
            ),
        }
    )

generated_df = pd.DataFrame(
    generated_news
)

print(
    generated_df[
        [
            "№",
            "generated",
            "ends_correctly",
        ]
    ].to_string(
        index=False
    )
)

 №                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           generated  ends_correctly
 1                                                                                                                                                                                                                                               

In [40]:
# Проверка окончания

all_finished = all(
    row[
        "ends_correctly"
    ]
    for row in generated_news
)

print(
    "Все новости завершены "
    "знаком . ! или ?:",
    all_finished,
)

assert all_finished

Все новости завершены знаком . ! или ?: True


In [41]:
# Сохранение

SAVE_DIR = (
    "/content/"
    "rugpt_medical_news_final"
)

trainer.save_model(
    SAVE_DIR
)

tokenizer.save_pretrained(
    SAVE_DIR
)

print(
    "Модель сохранена:",
    SAVE_DIR
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель сохранена: /content/rugpt_medical_news_final


Русскоязычная GPT-модель была дообучена на медицинских новостях.

Для генерации были проверены разные режимы `model.generate`. Итоговый вариант использует сочетание **temperature + top-k + top-p**, а также штраф за повторы.

Итогом являются 10 учебных синтетических новостей на тему медицины и медицинского образования.